# Summary Stats on a Proposed Rebalance

In [72]:
import json
import pandas as pd



path =  '../data/rebalance_plan_1720826447.json' # july 12, 2024

In [104]:
def make_in_out_apr_df(state_of_destinations) -> pd.DataFrame:
    df = pd.DataFrame()
    df['name'] = [dest['name'] for dest in state_of_destinations['destStates']]
    df['totalAprIn'] = [dest['totalAprIn'] for dest in state_of_destinations['destStates']]
    df['totalAprOut'] = [dest['totalAprOut'] for dest in state_of_destinations['destStates']]
    df['destination'] = [dest['address'] for dest in state_of_destinations['destStates']]
    df['lp_safePrice'] = [dest['spotPrice'] for dest in state_of_destinations['destStates']]
    df['lp_spotPrice'] = [dest['safePrice'] for dest in state_of_destinations['destStates']]
    return df


def build_summary_data(path:str) -> dict:
    
    with open(path, 'r') as fin: 
        data = json.load(fin)
    
    
    state_of_destinations = data.pop('sod')
    apr_df = make_in_out_apr_df(state_of_destinations)
    
    destinationOut_name = apr_df[apr_df['destination'] == data['destinationOut']]['name'].values[0]
    destinationIn_name = apr_df[apr_df['destination'] == data['destinationIn']]['name'].values[0]
    
    totalAprIn = float(apr_df[apr_df['destination'] == data['destinationIn']]['totalAprIn'].values[0] * 100)
    totalAprOut = float(apr_df[apr_df['destination'] == data['destinationOut']]['totalAprOut'].values[0] * 100)
    
    amountOutETH = int(data['amountOutETH']) / 1e18
    minAmountInETH = int(data['minAmountIn']) / 1e18
    
    expectedAPRIncrease = totalAprIn - totalAprOut
    slippage_percent = 100 - ((100 * int(data['minAmountInETH'])) / int(data['amountOutETH']))
    
    one_year_of_earnings_if_no_change = (totalAprOut / 100)  * amountOutETH
    one_year_of_earnings_if_move = (totalAprIn / 100)  * minAmountInETH
    real_eth_cost_to_move = amountOutETH - minAmountInETH
    
    expected_increase_in_a_year = one_year_of_earnings_if_move - real_eth_cost_to_move
    
    summary = {
        'destinationOut_name':destinationOut_name, 'destinationIn_name':destinationIn_name,
        'totalAprOut':round(totalAprOut,2 ), 'totalAprIn':round(totalAprIn,2),
        'amountOutETH':amountOutETH, 'minAmountInETH':minAmountInETH, 
        'expectedAPRIncrease': expectedAPRIncrease ,'slippagePercent': round(slippage_percent, 2),
        'one_year_of_earnings_if_no_change': one_year_of_earnings_if_no_change,
        'one_year_of_earnings_if_move': one_year_of_earnings_if_move, 
        'expected_increase_in_a_year':expected_increase_in_a_year,
        'real_eth_cost_to_move': real_eth_cost_to_move
    }
    # return data
    return apr_df, summary

apr_df, summary = build_summary_data(path)
summary

{'destinationOut_name': 'Tokemak-Wrapped Ether-osETH/rETH',
 'destinationIn_name': 'Tokemak-Wrapped Ether-Balancer ETHx/wstETH',
 'totalAprOut': 6.85,
 'totalAprIn': 10.77,
 'amountOutETH': 8.912144008546347,
 'minAmountInETH': 8.804791581051274,
 'expectedAPRIncrease': 3.9246051076593558,
 'slippagePercent': 0.46,
 'one_year_of_earnings_if_no_change': 0.6102902611983722,
 'one_year_of_earnings_if_move': 0.9484922280079886,
 'real_eth_cost_to_move': 0.10735242749507279}

In [96]:
apr_df

,name,totalAprIn,totalAprOut,destination,lp_safePrice,lp_spotPrice
0,Tokemak-Wrapped Ether-Curve.fi ETH/stETH,0.017778,0.017871,0x75FD0d0247fA088852417CD0F1bfa21D1d78aa14,1.097503,1.097502
1,Tokemak-Wrapped Ether-Curve.fi Factory Pool: s...,0.021454,0.023174,0xD43e6d2a8B983DDEf52eC50eF0E3159542fEF8ed,1.031541,1.031584
2,Tokemak-Wrapped Ether-Curve.fi Factory Crypto ...,0.029464,0.029464,0x1A73e18B2a677940Cf5d5eb8bC244854Dc07d551,2.104032,2.103996
3,Tokemak-Wrapped Ether-osETH/rETH,0.068479,0.068479,0x772C047f317381c8F2DBd7B43E13B704EfFdDD45,1.024717,1.024699
4,Tokemak-Wrapped Ether-Curve.fi Factory Plain P...,0.028063,0.028103,0x8bf50aA240564bC079ffB2a94265e471509c7163,1.140478,1.140564
5,Tokemak-Wrapped Ether-Balancer stETH Stable Pool,0.010405,0.010405,0x38e73E98d2038FafdC847F13dd9100732383B6F2,1.047029,1.047030
6,Tokemak-Wrapped Ether-Balancer rETH Stable Pool,0.023796,0.057237,0xfb1f48a461cCC70081226d8353e45CfBd410dD8F,1.037789,1.037818
7,Tokemak-Wrapped Ether-Curve.fi Factory Pool: E...,0.091173,0.091589,0x2E5A8C3aE475734Ece6443B5E68F7fA63133AF3D,1.021511,1.021479
8,Tokemak-Wrapped Ether-weETH/WETH-ng,0.045461,0.051815,0x1568528A93393B8dfc708f3FD0537C549290AC73,1.016773,1.016774
9,Tokemak-Wrapped Ether-Balancer rsETH / ETHx,0.100202,0.100336,0x37e565f997c2b16d2542E906672E9c6281e77954,1.011114,1.011127


In [ ]:
# question, did this actually occur

In [60]:
steps[-1]

{'stepType': 'addLiquidity',
 'poolType': 'balCompStable',
 'poolAddress': '0xB91159aa527D4769CB9FAf3e4ADB760c7E8C8Ea7',
 'lpTokenAddress': '0xB91159aa527D4769CB9FAf3e4ADB760c7E8C8Ea7',
 'tokens': ['0xA35b1B31Ce002FBF2058D22F30f95D405200A15b',
  '0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0'],
 'amounts': ['4858710532811623424', '3282723021510579456'],
 'minLpAmountOut': '0'}

In [ ]:
# go from a 6.8% toa 10.7%

In [52]:
data

{'timestamp': 1720826447,
 'chainId': 1,
 'solverAddress': '0x2C26808b567BA224652f4eB20D45df4bccC29470',
 'poolAddress': '0x49C4719EaCc746b87703F964F09C22751F397BA0',
 'destinationOut': '0x772C047f317381c8F2DBd7B43E13B704EfFdDD45',
 'tokenOut': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'amountOut': '8697325088981711872',
 'amountOutETH': '8912144008546348032',
 'destinationIn': '0xa3956D49106288E5c04E6FBbBad5b68593f0bE3b',
 'tokenIn': '0xB91159aa527D4769CB9FAf3e4ADB760c7E8C8Ea7',
 'minAmountIn': '8804791581051274240',
 'minAmountInETH': '8870929682676062208',
 'steps': [{'stepType': 'removeLiquidity',
   'poolType': 'balCompStable',
   'poolAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
   'lpTokenAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
   'lpAmountOut': '8697325088981711872',
   'tokens': ['0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38',
    '0xae78736Cd615f374D3085123A210448E74Fc6393'],
   'minTokenAmounts': ['5313205317771941888', '3098986984605414400']},


In [ ]:
destinationOut = data['destinationOut']
destinationIn = data['destinationIn']

In [42]:
steps = data['steps']

steps[0]

{'stepType': 'removeLiquidity',
 'poolType': 'balCompStable',
 'poolAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'lpTokenAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'lpAmountOut': '8697325088981711872',
 'tokens': ['0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38',
  '0xae78736Cd615f374D3085123A210448E74Fc6393'],
 'minTokenAmounts': ['5313205317771941888', '3098986984605414400']}

In [43]:
steps[-1]

{'stepType': 'addLiquidity',
 'poolType': 'balCompStable',
 'poolAddress': '0xB91159aa527D4769CB9FAf3e4ADB760c7E8C8Ea7',
 'lpTokenAddress': '0xB91159aa527D4769CB9FAf3e4ADB760c7E8C8Ea7',
 'tokens': ['0xA35b1B31Ce002FBF2058D22F30f95D405200A15b',
  '0x7f39C581F595B53c5cb19bD0b3f8dA6c935E2Ca0'],
 'amounts': ['4858710532811623424', '3282723021510579456'],
 'minLpAmountOut': '0'}

('8912144008546348032', '8870929682676062208')

In [51]:
data

{'timestamp': 1720826447,
 'chainId': 1,
 'solverAddress': '0x2C26808b567BA224652f4eB20D45df4bccC29470',
 'poolAddress': '0x49C4719EaCc746b87703F964F09C22751F397BA0',
 'destinationOut': '0x772C047f317381c8F2DBd7B43E13B704EfFdDD45',
 'tokenOut': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'amountOut': '8697325088981711872',
 'amountOutETH': '8912144008546348032',
 'destinationIn': '0xa3956D49106288E5c04E6FBbBad5b68593f0bE3b',
 'tokenIn': '0xB91159aa527D4769CB9FAf3e4ADB760c7E8C8Ea7',
 'minAmountIn': '8804791581051274240',
 'minAmountInETH': '8870929682676062208',
 'steps': [{'stepType': 'removeLiquidity',
   'poolType': 'balCompStable',
   'poolAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
   'lpTokenAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
   'lpAmountOut': '8697325088981711872',
   'tokens': ['0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38',
    '0xae78736Cd615f374D3085123A210448E74Fc6393'],
   'minTokenAmounts': ['5313205317771941888', '3098986984605414400']},


In [53]:
data

{'timestamp': 1720826447,
 'chainId': 1,
 'solverAddress': '0x2C26808b567BA224652f4eB20D45df4bccC29470',
 'poolAddress': '0x49C4719EaCc746b87703F964F09C22751F397BA0',
 'destinationOut': '0x772C047f317381c8F2DBd7B43E13B704EfFdDD45',
 'tokenOut': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'amountOut': '8697325088981711872',
 'amountOutETH': '8912144008546348032',
 'destinationIn': '0xa3956D49106288E5c04E6FBbBad5b68593f0bE3b',
 'tokenIn': '0xB91159aa527D4769CB9FAf3e4ADB760c7E8C8Ea7',
 'minAmountIn': '8804791581051274240',
 'minAmountInETH': '8870929682676062208',
 'steps': [{'stepType': 'removeLiquidity',
   'poolType': 'balCompStable',
   'poolAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
   'lpTokenAddress': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
   'lpAmountOut': '8697325088981711872',
   'tokens': ['0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38',
    '0xae78736Cd615f374D3085123A210448E74Fc6393'],
   'minTokenAmounts': ['5313205317771941888', '3098986984605414400']},


To APR, from APR, 

swap cost, expected gain / swap cost offset period